# Portfolio 3 — Multi-Agent Reinforcement Learning

| | |
|---|---|
| **Project** |Multi-Agent Reinforcement Learning |
| **Omgeving** | `pettingzoo.atari.warlords_v3`|
| **Algoritme** | PPO-Clip met HEPPO-uitbreidingen, self-play met ELO-pool |

> In dit project trainen wij een reinforcement-learning agent voor het Atari-spel Warlords. In dit spel zijn er vier spelers die elk hun eigen fort verdedigen.

> Wij gebruiken PPO (Proximal Policy Optimization) als basisalgoritme. Daarnaast breiden wij dit uit met stabiliteitstechnieken uit HEPPO, zodat het trainen betrouwbaarder verloopt.

>De agent wordt getraind via self-play. Daarbij speelt hij tegen eerdere versies van zichzelf en tegen andere tegenstanders uit een pool die wordt gewogen met ELO, een systeem om de sterkte van spelers te vergelijken.

> Het doel van het project is om een agent te ontwikkelen die goed kan presteren in een multi-agent omgeving. Uiteindelijk moet de agent sterk genoeg zijn om mee te doen aan een toernooi met de hele klas.


## Inhoudsopgave

1. [Inleiding](#inleiding)
2. [Probleemstelling](#probleemstelling)
3. [Doel en onderzoeksvragen](#doelstelling)
4. [Algoritme kiezen](#Algoritmekiezen)
- 4.1 [De omgeving: *Warlords*](#omgeving)
- 4.2 [Waarom PPO?](#ppo)
- 4.3 [Netwerkarchitectuur: gedeelde MLP over RAM](#netwerk)
- 4.4 [Voordelen schatten met Generalized Advantage Estimation](#gae)
- 4.5 [PPO stabieler maken met HEPPO](#heppo)
- 4.6 [Het late-beloningsprobleem](#beloning)
- 4.7 [Self-play met een tegenstanderspool](#selfplay)
- 4.8 [ELO als maat voor voortgang](#elo)
5. [Implementatie](#Implementatie)
- 5.1 [Baseline](#baseline)
- 5.2 [Packages en mappenstructuur](#packages)
- 5.3 [Rollout-buffer](#buffer)
- 5.4 [Testen zonder Atari-ROMs](#testing)
6. [Optimalisatie en evaluatie](#Optimalisatie)
- 6.1 [Hyperparameters](#hyperparameters)
- 6.2 [Ablations](#ablation)
- 6.3 [Voordeelstandaardisatie](#voordeelstandaardisatie)
7. [Conclusie](#conclusie)
8. [Reflectie en uitbreidingen](#reflectie)
- [Referenties](#referenties)


<a id="inleiding"></a>
## 1. Inleiding

Reinforcement Learning (RL) is een tak van machine learning waarin een agent leert door te handelen in een omgeving en daar feedback op te krijgen (Sutton & Barto, 2018). In de afgelopen jaren is RL sterk gegroeid, dankzij de combinatie met diepe neurale netwerken (Deep RL). Bekende successen zijn AlphaGo (Silver et al., 2018) en OpenAI Five in Dota 2 (Berner et al., 2019). De meeste van deze successen spelen zich af in omgevingen met één agent of met vaste teams. Multi-Agent Reinforcement Learning (MARL), waarbij meerdere agenten tegelijk leren en elkaar beïnvloeden, is veel complexer, omdat de omgeving voor elke agent voortdurend verandert doordat de andere agenten ook leren (Li et al., 2025).

Dit portfolio-project gaat precies over dat probleem. Wij bouwen een agent voor het Atari-spel Warlords, dat beschikbaar is in de PettingZoo-bibliotheek (Terry et al., 2021). Warlords is een spel met vier spelers die elk een fort verdedigen. De beloning is eenvoudig: +1 bij winst, −1 bij verlies en 0 voor alle tussenliggende stappen. Deze late en schaarse beloning maakt het leren moeilijk. Daarnaast trainen wij via self-play: de agent speelt steeds tegen eerdere versies van zichzelf, wat extra aandacht vraagt voor de stabiliteit van het leerproces (Berner et al., 2019).

In deze notebook leggen wij stap voor stap uit welke keuzes wij hebben gemaakt bij het ontwerpen van de agent. Wij beginnen met de probleemstelling en doelstelling. Daarna volgen de ontwerpkeuzes: per keuze beschrijven wij het probleem, de mogelijke opties, de uiteindelijke beslissing, de motivatie en de plek in de code. De volgorde volgt het echte verloop van het project, niet een theoretisch ideale structuur, maar het pad dat wij in de praktijk hebben doorlopen.


<a id="probleemstelling"></a>
## 2. Probleemstelling

Het hoofdprobleem van dit project is: hoe trainen wij een enkele RL-agent die competitief kan meedoen in een multi-agent Atari-omgeving (Warlords) met vier spelers, schaarse terminale beloningen (+1/−1) en niet-stationaire tegenstanders?

Dit hoofdprobleem valt uiteen in vijf deelproblemen, die elk in een eigen sectie van de ontwerpnotities worden behandeld:

1.	Algoritmekeuze. Welk Deep RL-algoritme is geschikt voor een discrete, multi-agent en gedeeltelijk observeerbare Atari-omgeving? En hoe gaan wij om met het feit dat vier agenten tegelijk leren?

2.	Numerieke stabiliteit. PPO staat erom bekend gevoelig te zijn voor de schaal van beloningen en waarden (Engstrom et al., 2020). Hoe houden wij de training stabiel, zeker met de ±1-beloning van Warlords?

3.	Credit assignment over lange episodes. De beloning komt pas aan het einde, soms na honderden stappen. Hoe zorgen wij ervoor dat de agent leert welke acties hebben bijgedragen aan winst of verlies?

4.	Stabiele self-play. Wanneer alle vier de spelers dezelfde agent zijn, trainen wij in feite tegen onszelf. Dit kan leiden tot moving-target instabiliteit en strategie-cycling (Heinrich & Silver, 2016). Hoe bouwen wij een self-play-opzet die stabiel blijft?

5.	Eerlijke voortgangsmeting. Binnen self-play betekent een stijgende return-curve niet automatisch dat de agent beter wordt, omdat de tegenstanders ook veranderen. Hoe meten wij echte vooruitgang?

Deze vijf deelproblemen zijn niet los van elkaar te zien. De keuze voor PPO  beïnvloedt bijvoorbeeld hoe wij stabiliteit aanpakken , en de keuze voor self-play bepaalt welke evaluatiemetriek betrouwbaar is . Daarom volgen de ontwerpnotities niet een strak theoretisch schema, maar de volgorde van het daadwerkelijke ontwikkelproces.


<a id="doelstelling"></a>
## 3. Doelstelling

<a id="doelstelling"></a>
Het doel van dit project is om een werkende, reproduceerbare en goed reinforcement-learning agent te bouwen voor de multi-agent Atari-omgeving Warlords. De agent moet na training beter presteren dan een random en een simpele rule-based baseline, en dus echt leerbaar gedrag laten zien in een competitieve setting.

<a id="Algoritmekiezen"></a>
# 4. Algoritme kiezen

<a id='omgeving'></a>
## 4.1 De omgeving: *Warlords*

**Omgeving:** pettingzoo.atari.warlords_v3 met RAM-waarnemingen (Terry et al., 2021).

-	Het spel bestaat uit 4 spelers die allemaal dezelfde rol hebben en elk een fort verdedigen.
-	De waarneming per agent is een 128-byte RAM-vector. Deze representatie komt uit de Arcade Learning Environment, waarin de interne emulator-RAM wordt blootgesteld als compacte toestandsbeschrijving (Bellemare et al., 2013).
-	De actieruimte bestaat uit 6 discrete acties.
-	Beloning (cruciaal voor de rest van het ontwerp): elke tijdstap levert een beloning van 0 op. Pas aan het einde van de episode wordt een beloning van +1 (winst) of −1 (verlies) toegekend. Episodes zijn lang en bestaan vaak uit honderden stappen.
-	De omgeving is competitief en multi-agent van aard en kan worden benaderd als een (bijna) nul-som setting (Farama Foundation, 2024).



<a id="ppo"></a>
### 4.2 Waarom PPO?

Wij hebben een deep reinforcement learning-algoritme nodig dat een stochastisch beleid kan leren in een Atari-omgeving met discrete acties en meerdere agenten.

| Familie | Voorbeeld | Voordelen | Nadelen voor ons |
|---|---|---|---|
| Waarde-gebaseerd | **DQN**, Double-DQN, Rainbow | sample-efficiënt, veel getest op Atari (Mnih et al., 2015) | off-policy met replay-buffer is lastig onder self-play (oude tegenstanders in buffer); ε-greedy is zwak in competitief spel; geen stochastisch beleid |
| On-policy actor-critic | **A2C / A3C** (Mnih et al., 2016) | eenvoudig, stabiel | geen trust-region → grotere updates kunnen het beleid laten instorten bij ruis en schaarse beloning |
| Trust-region | **TRPO** (Schulman et al., 2015) | sterke theoretische garanties | tweede-orde; duur; lastig met parameter sharing |
| Clipped policy gradient | **PPO** (Schulman et al., 2017) | eerste-orde, eenvoudig, stochastisch beleid, goed werkend op Atari en in multi-agent | gevoelig voor hyperparameters; vraagt zorg met beloningsschaal (Engstrom et al., 2020) |
| Off-policy actor-critic (continu) | SAC, DDPG, TD3 | sample-efficiënt | gemaakt voor continue acties; geen natuurlijke fit voor Atari |
| Model-based / planning | MuZero, Dreamer | erg sterk op Atari | veel complexer; overkill voor dit project |

Wij kiezen voor PPO-Clip als basisalgoritme.

-	PPO is een standaardkeuze binnen on-policy deep reinforcement learning vanwege de combinatie van eenvoud, stabiliteit en brede toepasbaarheid in multi-agent self-play omgevingen. Het vormt ook de basis voor systemen zoals AlphaStar en OpenAI Five (Berner et al., 2019).
-	PPO ondersteunt van nature stochastische beleidsvorming. Dit is essentieel in self-play, omdat deterministische policies kwetsbaar zijn voor exploitatie en kunnen leiden tot cyclische dynamiek (Hui, 2018).
-	Omdat PPO on-policy is, worden updates altijd uitgevoerd op data die is verzameld tegen de actuele tegenstanders. Hierdoor wordt het probleem van verouderde ervaringen in replay buffers vermeden, wat met name belangrijk is in een dynamisch self-play regime (Berner et al., 2019).


<a id="netwerk"></a>
### 4.3 Netwerkarchitectuur: gedeelde MLP over RAM

Een belangrijke ontwerpkeuze is de architectuur van het actor-critic netwerk.

**Opties.**


- **Convolutionele netwerken (CNN’s)** over 210×160×3 pixel-invoer, zoals in klassieke Atari-architecturen (Mnih et al., 2015).
-  **Multi-Layer Perceptrons (MLP’s)** over de 128-byte RAM-representatie van de omgeving (Anschel et al., 2017).
-  **Recurrente architecturen**, zoals LSTM- of GRU-netwerken, om temporele afhankelijkheden expliciet te modelleren.
-  **Verschillende parametrisatiekeuzes:** aparte netwerken per agent versus één gedeeld netwerk.

Wij gebruiken een compacte **MLP** die werkt op de 128-byte RAM-vector. Het netwerk bestaat uit een gedeelde trunk met twee aparte outputheads: één policy-head die logits produceert over 6 discrete acties en één value-head die een scalaire waardeschatting geeft. Deze architectuur wordt gedeeld over alle vier de agent-slots.

-  De RAM-representatie bevat reeds een gecomprimeerde beschrijving van de speltoestand, waardoor visuele feature-extractie via CNN’s overbodig is. Dit leidt tot een efficiënter en sneller leerproces (Anschel et al., 2017).
-  Omdat de vier spelers in *Warlords* symmetrisch zijn, ligt parameter sharing voor de hand. Door één gedeeld netwerk te gebruiken wordt de sample-efficiëntie verhoogd en neemt de benodigde hoeveelheid data ruwweg af met een factor gelijk aan het aantal agenten (Gupta et al., 2017).

Het netwerk voldoet aan de vereisten voor een actor-critic architectuur met gedeelde trunk en gescheiden policy- en value-heads. De kerncomponenten van de PPO-update (policy loss, value loss en entropy-regularisatie) zijn geïmplementeerd in `ppo_agent.py`. In tegenstelling tot off-policy methoden maakt PPO geen gebruik van een replay buffer, maar van een rollout buffer.

<a id="gae"></a>
### 4.4 Voordelen schatten met Generalized Advantage Estimation

Voor het uitvoeren van PPO-updates is een schatting van het voordeel $\hat{A}_t$ nodig. Deze grootheid geeft aan hoeveel beter of slechter een uitgevoerde actie was ten opzichte van wat de waardefunctie verwachtte. Een belangrijke ontwerpkeuze is daarom hoe deze voordelen worden berekend.

**Opties.**
- Monte-Carlo: returns Deze methode levert onbevooroordeelde schattingen op, maar gaat gepaard met een hoge variantie, wat het leerproces instabiel kan maken (Sutton & Barto, 2018).
- One-step TD: Deze aanpak heeft een lagere variantie, maar introduceert meer bias in de schattingen.
- **GAE**: Een methode die een gewogen combinatie vormt van n-step returns en daarmee een expliciete afweging mogelijk maakt tussen bias en variantie via de parameter $\lambda$ $\lambda$ (Schulman et al., 2016).

Wij hebben gekozen voor Generalized Advantage Estimation (GAE), waarbij zowel $\gamma$ als $\lambda$ als afzonderlijke hyperparameters worden gebruikt. De voordelen worden berekend met behulp van de standaard recursieve formulering:

$$\hat{A}_t = \delta_t + (\gamma\lambda)\,\hat{A}_{t+1},\qquad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t).$$

-	GAE is de standaardmethode voor voordeelsschatting binnen PPO en wordt toegepast in de oorspronkelijke PPO-publicatie (Schulman et al., 2017).
-	De methode biedt een effectieve balans tussen bias en variantie, waardoor de resulterende gradiëntschattingen stabieler zijn dan bij zuivere Monte-Carlo- of TD-methoden.
-	De parameter $\lambda$ geeft ons extra controle over de effectieve tijdshorizon van de voordeelsschattingen. Deze flexibiliteit is met name belangrijk in Warlords, waar beloningen vaak pas aan het einde van lange episodes worden toegekend. In Sectie 4.6 maken wij gebruik van deze eigenschap om het probleem van vertraagde beloningen beter aan te pakken.

Op basis van deze overwegingen vormt GAE een natuurlijke keuze voor onze PPO-implementatie.

<a id="heppo"></a>
### 4.5 PPO stabieler maken met HEPPO

Tijdens de eerste experimenten met een standaard PPO-implementatie op Warlords observeerden wij verschillende bekende instabiliteiten. Het waardeverlies domineerde regelmatig de totale verliesfunctie en vroege beloningen hadden een onevenredig grote invloed op de trainingsstatistieken. Dergelijke problemen zijn eerder beschreven als veelvoorkomende faalmodi van PPO (Engstrom et al., 2020).

HEPPO als inspiratiebron. Tijdens ons literatuuronderzoek hebben wij HEPPO bestudeerd (Taha & Abdelhadi, 2025). Hoewel dit werk primair gericht is op een FPGA-gebaseerde accelerator voor PPO, bevat Sectie II een aantal algoritmische technieken die onafhankelijk van de hardwarearchitectuur toepasbaar zijn. Wij hebben drie van deze technieken geïntegreerd in onze software-implementatie:


1. **Dynamische beloningsstandaardisatie** op basis van Welfords online algoritme voor het bijhouden van gemiddelde en variantie (Welford, 1962).
2. **Blok-standaardisatie van waarden** per rollout, gevolgd door de-standaardisatie voordat deze waarden worden gebruikt binnen Generalized Advantage Estimation.
3. **Uniforme n-bit kwantisatie** van gestandaardiseerde waarden.

De hardware-specifieke optimalisaties uit HEPPO hebben wij niet overgenomen.

Wij hebben alle drie de algoritmische technieken uit HEPPO geïntegreerd in onze PPO-implementatie om de stabiliteit van het leerproces te verbeteren.

- Beloningsstandaardisatie lost het bekende probleem op dat PPO gevoelig is voor beloningsschaal (Engstrom et al., 2020).
- Waarde-standaardisatie houdt het waardeverlies op dezelfde schaal als de policy-loss. Anders is `vf_coef` een fragiele hyperparameter (Huang et al., 2022).
- Kwantisatie is vooral een hardware-truc, maar werkt ook als milde regularisator (Taha & Abdelhadi, 2025).

Door deze technieken te combineren verwachten wij een stabielere PPO-training, waarbij zowel de policy-update als de waardeschatting beter beheersbaar blijven gedurende lange trainingsruns.

In [ ]:
# Sanity-check Welford's online stats against the closed-form mean/std.
import numpy as np

rng = np.random.default_rng(0)
stream = rng.normal(loc=3.0, scale=2.0, size=10_000)

n, mean, M2 = 0, 0.0, 0.0
for x in stream:
    n += 1
    delta = x - mean
    mean += delta / n
    M2 += delta * (x - mean)

var = M2 / (n - 1)
print(f'Welford : mean={mean:.6f}  std={np.sqrt(var):.6f}')
print(f'NumPy   : mean={stream.mean():.6f}  std={stream.std(ddof=1):.6f}')

<a id="beloning"></a>
### 4.6 Het late-beloningsprobleem

`warlords_v3` geeft elke stap 0. Pas op het einde komt +1 of −1 (Farama Foundation, 2024). Episodes zijn lang: honderden stappen. Met de standaard PPO-default `gamma=0.99` is dit signaal vrijwel onzichtbaar. Een beloning 500 stappen terug draagt nog $0.99^{500} \approx 0.0066$ bij aan de waarde op tijdstip $t$ — dat verdwijnt in de ruis (Sutton & Barto, 2018).

**Opties.**
1. **Reward shaping**: aanvullende tussentijdse beloningen introduceren om een dichter leersignaal te creëren.
2. **Curiosity / intrinsieke motivatie**: technieken zoals Random Network Distillation (RND) of Intrinsic Curiosity Modules (ICM).
3. **Grotere $\gamma$ + grotere $\lambda$**: de effectieve credit-assignment-horizon vergroten door de discontovoet en GAE-parameter te verhogen (Schulman et al., 2016).
4. **Hindsight / return-conditioning**: methoden die expliciet gebruikmaken van uiteindelijke uitkomsten bij het leren.

Wij hebben gekozen voor **optie 3** en verhogen de standaardwaarden naar $\gamma$ = 0.999 en $\lambda$ = 0.97. Daarnaast hebben wij een uitbreidingspunt voor reward shaping opgenomen in de implementatie, maar deze functionaliteit staat standaard uitgeschakeld.

-	Reward shaping is in competitieve win/verlies-omgevingen risicovol. De agent kan leren om het surrogaatsignaal te optimaliseren in plaats van het werkelijke doel, namelijk het winnen van de wedstrijd. Een agent kan bijvoorbeeld leren om zoveel mogelijk stenen te raken zonder dat dit daadwerkelijk leidt tot betere prestaties. Ng et al. (1999) beschrijven voorwaarden waaronder reward shaping beleidsinvariant blijft; de door ons overwogen shaping-signalen voldoen niet aan deze voorwaarden.
-	Curiosity-methoden richten zich voornamelijk op exploratie. In onze omgeving is exploratie echter niet de belangrijkste beperking; het centrale probleem is het correct toewijzen van een eindbeloning aan acties die honderden stappen eerder zijn uitgevoerd.
-	Het verhogen van $\gamma$ pakt dit probleem direct aan door toekomstige beloningen zwaarder mee te laten wegen. Een hogere waarde van $\lambda$ vergroot daarnaast de effectieve horizon van Generalized Advantage Estimation, waarbij meer bias wordt ingeruild voor een lagere variantie van de schattingen.
-	Het effect van deze keuze is aanzienlijk. Voor een episode van 500 stappen geldt dat $0.999^{500} \approx 0.61$, terwijl $0.99^{500} \approx 0.0066$. Het leersignaal blijft daardoor ongeveer negentig keer sterker behouden, waardoor de eindbeloning veel effectiever kan worden teruggevoerd naar eerdere beslissingen.

Op basis van deze opties verwachten wij dat een verhoogde waarde van $\gamma$ de meest directe en principiële oplossing biedt voor het late-beloningsprobleem in Warlords.


In [ ]:
# How far back in time does a single terminal reward 'reach'?
import numpy as np
import matplotlib.pyplot as plt

T = 1000
t = np.arange(T)
for gamma in (0.99, 0.995, 0.999):
    plt.plot(t, gamma ** t, label=f'gamma={gamma}')
plt.axhline(0.01, color='gray', linestyle='--', label='1% of peak')
plt.xlabel('steps before terminal reward')
plt.ylabel('discounted contribution')
plt.yscale('log')
plt.legend()
plt.title('Effective credit-assignment horizon as a function of gamma')
plt.show()

<a id="selfplay"></a>
### 4.7 Self-play met een tegenstanderspool

De omgeving Warlords bevat vier agent-slots. De meest eenvoudige aanpak is om één gedeeld netwerk alle vier de slots te laten besturen. In een nul-somomgeving met beloningen van ±1 leidt dit echter tot een fundamenteel probleem: hetzelfde beleid ontvangt tegelijkertijd signalen dat een traject succesvol én onsuccesvol was. Dit staat bekend als de nul-som-tegen-zichzelf-degeneratie (Heinrich & Silver, 2016).

**Overwogen opties**

-	Naïeve parameter sharing. Eén gedeeld netwerk bestuurt alle vier de agenten en wordt op alle trajecten getraind (Gupta et al., 2017).
-	Onafhankelijke policies. Elke agent krijgt een eigen netwerk. Deze aanpak verviervoudigt echter het aantal parameters en benut de symmetrie van het probleem niet.
-	Self-play met een vaste tegenstander. Eenvoudig te implementeren, maar gevoelig voor overfitting aan één specifieke speelstijl.
-	Self-play met een tegenstanderpool. Eén lerende agent speelt tegen een verzameling van eerdere versies van zichzelf die als tegenstanders fungeren (Berner et al., 2019).
-	League-training of Prioritized Fictitious Self-Play (PFSP). Een uitgebreide league-structuur zoals toegepast in AlphaStar (Vinyals et al., 2019). Hoewel krachtig, is deze aanpak aanzienlijk complexer dan nodig voor dit project.

Wij hebben gekozen voor self-play met een snapshot-pool (opponent_pool.py, train_warlords_selfplay.py). In elke omgeving is één agent de trainee die actief leert, terwijl de overige drie slots worden gevuld met bevroren snapshots uit een pool van eerder opgeslagen beleidsversies.

-	Self-play met snapshots is een bewezen methode om competitieve reinforcement-learningagenten stabiel te trainen en wordt toegepast in systemen zoals AlphaZero (Silver et al., 2018).
-	Door tegen eerdere versies van zichzelf te spelen, leert de agent robuuste strategieën in plaats van zich uitsluitend aan te passen aan de meest recente tegenstander.
-	Het gebruik van een tegenstanderpool vermindert bovendien het risico op catastrofisch vergeten, omdat de agent regelmatig geconfronteerd blijft worden met oudere speelstijlen (Berner et al., 2019).

**Samplingstrategie**

Binnen de tegenstanderpool gebruiken wij ELO-gewogen sampling met een temperatuurparameter. Hierdoor worden sterkere snapshots vaker geselecteerd, terwijl er voldoende variatie behouden blijft. Wanneer de pool zijn maximale grootte bereikt, verwijderen wij steeds de oudste snapshot in plaats van de zwakste. Op deze manier blijft de pool een representatieve afspiegeling van de historische ontwikkeling van het beleid en behouden wij voldoende strategische diversiteit (DeepMind, 2019).




<a id="elo"></a>
### 4.8 ELO als maat voor voortgang

Om de prestaties van onze agent te evalueren, moeten wij bepalen hoe we kunnen vaststellen of de agent daadwerkelijk sterker wordt gedurende het trainingsproces.

**Uitdaging**. Bij self-play verandert de verdeling van tegenstanders voortdurend, omdat de agent steeds vaker tegen eerdere versies van zichzelf speelt. Hierdoor kan de gemiddelde episode-return moeilijk te interpreteren zijn. Een stijgende return kan betekenen dat onze agent sterker wordt, maar kan ook het gevolg zijn van het spelen tegen relatief zwakkere tegenstanders binnen de trainingspool (Silver et al., 2018).

Daarom gebruiken wij de ELO-score van de trainee ten opzichte van de tegenstanderpool als primaire prestatiemaatstaf. Daarnaast volgen wij een rolling win-rate om recente prestaties inzichtelijk te maken. De ruwe episode-return wordt eveneens gelogd, maar krijgt een minder prominente rol in de analyse.

Een belangrijk voordeel van ELO is dat deze maat rekening houdt met de sterkte van de tegenstander. Hierdoor biedt de score een betrouwbaardere indicatie van relatieve speelsterkte dan de gemiddelde return alleen. Om die reden wordt ELO al decennialang gebruikt binnen de schaakwereld en wordt het eveneens toegepast in moderne self-play-systemen zoals AlphaZero en AlphaStar. Door ELO als primaire maatstaf te gebruiken, kunnen wij de voortgang van onze agent beter volgen, ook wanneer de kwaliteit van de tegenstanders gedurende de training verandert.

<a id="Implementatie"></a>
# 5. Implementatie


<a id="baseline"></a>
### 5.1 Baseline

Om te kunnen beoordelen of onze agent daadwerkelijk leert, hebben wij geschikte vergelijkingspunten nodig. De opdracht vereist het gebruik van een baseline, bijvoorbeeld een willekeurige of rule-based policy. Daarom hebben wij twee verschillende baselines geselecteerd.

Twee baselines:

1. **Random baseline.** Deze baseline kiest bij elke tijdstap willekeurig één van de zes beschikbare acties. Dit vormt het minimale prestatieniveau waar een lerende agent minstens bovenuit moet stijgen.
2. **Rule-based baseline.** Deze baseline maakt gebruik van een eenvoudige heuristiek die de bal volgt en het eigen fort verdedigt. De implementatie is gebaseerd op de voorbeeldcode die beschikbaar wordt gesteld binnen PettingZoo (Farama Foundation, 2024).

Door twee verschillende baselines te gebruiken, verkrijgen wij een vollediger beeld van de prestaties van onze agent. De random baseline laat zien of de agent beter presteert dan puur willekeurig gedrag, terwijl de rule-based baseline een vergelijking mogelijk maakt met een redelijke, handmatig ontworpen strategie. Hierdoor kunnen wij beter beoordelen in hoeverre de agent daadwerkelijk effectief gedrag heeft geleerd.

Beide baselines worden geëvalueerd binnen dezelfde multi-agentomgeving als onze RL-agent. Hierdoor zijn de resultaten direct vergelijkbaar en kunnen prestatieverschillen worden toegeschreven aan het gebruikte beleid in plaats van aan verschillen in de experimentele opzet.

<a id="packages"></a>
### 5.2 Packages


| Component | Package | Waarom |
|---|---|---|
| Multi-agent omgeving | `pettingzoo` + `AutoROM` |Wij gebruiken PettingZoo als standaardframework voor multi-agent reinforcement learning. De omgeving Warlords is hierin direct beschikbaar, waardoor wij een goed ondersteunde en reproduceerbare experimentele omgeving hebben (Terry et al., 2021).|
| Atari-backend | `ale-py` | Wij gebruiken ale-py als interface naar de Arcade Learning Environment, de standaardomgeving voor Atari-gebaseerd reinforcement learning onderzoek (Bellemare et al., 2013) |
| RL-algoritme | eigen implementatie | Wij hebben PPO zelf geïmplementeerd. Dit algoritme is compact genoeg om betrouwbaar zelf te ontwikkelen en geeft ons volledige controle over uitbreidingen en aanpassingen die nodig zijn voor HEPPO. |
| Tensors | `torch` | Wij gebruiken PyTorch vanwege de brede toepassing binnen deep reinforcement learning, de flexibiliteit van het framework en de goede ondersteuning voor GPU-versnelling. |
| Logging | `tensorboard` + `json` | Wij gebruiken TensorBoard voor het visualiseren van trainingscurves en prestatie-indicatoren. Daarnaast slaan wij resultaten op in JSON-formaat voor gestructureerde logging en latere analyse. |
| Plotting | `matplotlib` | Wij gebruiken Matplotlib voor het genereren van de figuren en grafieken die in dit rapport worden gepresenteerd. |


In [ ]:
# Quick layout check - what's in the repo?
import os
for root, dirs, files in os.walk('.', topdown=True):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    depth = root.count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for f in sorted(files):
        if f.endswith(('.py', '.md')):
            print('  ' * (depth + 1) + f)

<a id="buffer"></a>
### 5.3 Rollout-buffer

Voor het trainen van ons model verzamelen wij rollouts van $N$ parallelle omgevingen gedurende $T$ tijdstappen. Een belangrijke ontwerpkeuze is hoe wij deze gegevens opslaan.

**Opties.**
- Platte `(T·N,)` arrays.
- Omgeving-majeur `(N, T)`.
- Tijdstap-majeur `(T, N)`.

Wij hebben gekozen voor een tijdstap-majeure representatie met vorm `(T, N, ...)`.

**Waarom.**
- Generalized Advantage Estimation (GAE) wordt achterwaarts door de tijd berekend. Met een `(T, N)` -indeling kunnen wij deze recursie uitvoeren als één gevectoriseerde sweep over de tijdsdimensie (Schulman et al., 2016).
- De dones-indicatoren sluiten van nature aan op een structuur van vorm  `(T, N)`, wat de implementatie vereenvoudigt (Huang et al., 2022).
- Voor PPO-updates kunnen wij de gegevens vervolgens eenvoudig afvlakken naar `(T·N, ...)`, zodat minibatches efficiënt kunnen worden samengesteld (CleanRL; Huang et al., 2023).

Deze representatie biedt daarmee een goede balans tussen implementatiegemak, rekenkundige efficiëntie en compatibiliteit met de PPO-trainingsprocedure.

<a id="testing"></a>
### 5.4 Testen zonder Atari-ROMs

De sandbox kan de AutoROM-CDN niet bereiken, waardoor de Atari-ROMs tijdens CI- en ontwikkeltests niet beschikbaar zijn.

Daarom hebben wij ervoor gekozen om te testen met een mock van de parallelle API van PettingZoo (Terry et al., 2021), die de administratie van agent-eliminatie en episodebeëindiging nauwkeurig reproduceert. Alle overige componenten hebben wij getest met synthetische data die dezelfde structuur heeft als een Warlords-rollout (128-byte observaties, 6 acties).

Wij hebben daarbij de volgende tests uitgevoerd:

- Welfords statistieken vs. gesloten-vorm (Welford, 1962).
- Blok-standaardisatie round-trip (Taha & Abdelhadi, 2025).
- Kwantisatiefout schaalt met bitbreedte.
- GAE-recursie vs. referentie-implementatie (Schulman et al., 2016).
- ELO-updates: nul-som, upsets zwaarder gewogen.
- End-to-end PPO-update op een simpele bandiet-taak (actievoorkeur gaat van ~17% naar 100% in ~20 iteraties) (Schulman et al., 2017).

Op deze manier hebben wij alle componenten kunnen testen, met uitzondering van de Atari-ROM zelf. De bandiettest laat bovendien zien dat de PPO-update daadwerkelijk leert en het gewenste gedrag ontwikkelt.

<a id="Optimalisatie"></a>
# 6.  Optimalisatie en evaluatie

<a id="hyperparameters"></a>

### 6.1 Hyperparameters

Voor deze opdracht hebben wij geëxperimenteerd met verschillende hyperparameters, waaronder de learning rate en de balans tussen exploratie en exploitatie.

Wij hebben drie groepen experimenten uitgevoerd:

1. **Learning rate.** Drie waarden: `1e-4`, `3e-4` (default), `1e-3`. Te groot → instabiel; te klein → traag (Huang et al., 2022).
2. **Entropy-coëfficiënt.** Drie waarden: `0.0`, `0.01` (default), `0.05`. Houdt het beleid stochastisch genoeg voor exploratie (Hui, 2018).
3. **Diskontatiefactor $\gamma$.** Twee waarden: `0.99` (standaard) en `0.999`. Laat direct zien waarom de verhoogde $\gamma$ nodig is.

Voor elke configuratie hebben wij het model gedurende 1 miljoen stappen getraind en de resultaten vergeleken op basis van:

- Het ELO-traject (primaire maatstaf).
- De gemiddelde episode-return (secundaire maatstaf).
- De stabiliteit van het leerproces, gemeten als de standaarddeviatie van de ELO-score over de laatste 100.000 trainingsstappen.

<a id="ablation"></a>
### 6.2 Ablations

We hebben meerdere keuzes bovenop vanilla PPO gedaan: HEPPO, verhoogde $\gamma$, self-play, ELO-sampling. Welke daarvan doen er op *Warlords* echt toe?

`run_ablation.py` reproducert de 5 configuraties uit HEPPO's Table III (Taha & Abdelhadi, 2025), plus de toggles `--no-quantization`, `--no-adv-standardize`, `--quant-bits N`, `--use-reward-shaping`. Vergelijk op gemiddelde return voor het baseline-script en op **ELO-traject** voor het self-play-script.

**Waarom.** Beslissingen zonder ablations zijn slechts meningen. Herdraaien op *Warlords* laat zien of de beslissing overdraagt naar deze omgeving (Engstrom et al., 2020).

De ablation-harness toont aan dat we alle cruciale onderdelen van de DRL-pijplijn bewust hebben gevarieerd en gemeten.


<a id="voordeelstandaardisatie"></a>
### 6.3 Voordeelstandaardisatie

Zelfs met gestandaardiseerde beloningen en waarden kunnen de voordelen tussen rollouts in schaal verschillen.

Standaardiseer voordelen per minibatch naar gemiddelde 0 en std 1, vóór het PPO-clipped doel. Aan/uit via `--no-adv-standardize`.

**Waarom.**
- Dit is een vrijwel universele PPO-implementatiedetail (Huang et al., 2022).
- Het is orthogonaal aan de HEPPO-standaardisaties (Schulman et al., 2017).
- Houdbaar als ablatable knob.

<a id="conclusie"></a>

# 7. Conclusie

Wij hebben een reinforcement learning-agent ontwikkeld voor de multi-agent Atari-omgeving *Warlords*. De kern van ons ontwerp bestaat uit een reeks samenhangende keuzes die samen gericht zijn op stabiliteit, leerbaarheid en reproduceerbaarheid.

1.	**PPO-Clip** als algoritme: een eenvoudige en robuuste on-policy methode die goed werkt met stochastische beleidsvorming in competitieve settings.
2.	**Gedeelde MLP over de 128-byte RAM-vector**: een compacte architectuur die efficiënt gebruikmaakt van de symmetrie in de omgeving en snelle training mogelijk maakt.
3.	**HEPPO-gebaseerde standaardisaties**: technieken die de numerieke stabiliteit van zowel beloningen als waardeschattingen verbeteren.
4.	**Verhoogde parameters $\gamma = 0.999$ en $\lambda = 0.97$**: een bewuste keuze om het probleem van vertraagde beloningen te mitigeren zonder reward shaping toe te passen.
5.	**Self-play met snapshot-pool en ELO-gewogen sampling**: een aanpak die instabiliteit door een bewegende tegenstandersdistributie vermindert.
6.	**ELO als primaire prestatiemaatstaf**: een evaluatiemethode die corrigeert voor variërende tegenstanders en zo een eerlijker beeld geeft van de leerprogressie.

Gezamenlijk maken deze ontwerpkeuzes het mogelijk om stabiel te trainen in een competitieve multi-agent omgeving met sparse rewards en lange episodes. De volledige implementatie is gedocumenteerd, getest en reproduceerbaar.


<a id="reflectie"></a>
# 8. Reflectie en uitbreidingen


Het huidige project laat zien dat de gekozen aanpak functioneert, maar er blijven meerdere richtingen voor verbetering en uitbreiding.

**CNN’s over pixels in plaats van een MLP over RAM.**
   Hoewel trainen op RAM efficiënt is, kan een visuele representatie in sommige settings informatie bevatten die niet expliciet in de RAM-encoding aanwezig is. In een competitieve context kunnen andere agents mogelijk gebruikmaken van visuele cues die verloren gaan in de RAM-representatie (Anschel et al., 2017).

**Recurrente architecturen (LSTM/GRU).**
   Het toevoegen van recurrency kan helpen bij het modelleren van temporele afhankelijkheden en het onthouden van informatie die niet direct in de huidige observatie aanwezig is.

**League training in plaats van een tegenstanderpool.**
   Systemen zoals AlphaStar gebruiken een volledige league-structuur. Onze snapshot-pool is een vereenvoudigde variant hiervan. Een volledige league kan robuuster zijn, maar brengt aanzienlijk meer complexiteit met zich mee.

**Verantwoorde reward shaping.**
   Het toevoegen van potentiaal-gebaseerde shaping-functies kan het leerproces versnellen zonder het optimale beleid te veranderen, mits correct geformuleerd (Ng et al., 1999).

**Uitgebreidere hyperparameter-experimenten.**
   Verdere afstemming kan plaatsvinden op onder andere de clip-ratio, de waardeverlies-coëfficiënt, het aantal PPO-epochs en de grootte van de tegenstanderpool.

**Achteraf heroverwogen keuzes.**

* **Kwantisatie uit HEPPO.** Deze techniek is voornamelijk bedoeld voor hardware-optimalisatie en levert in software-implementaties beperkte voordelen op. In een volgende iteratie zou deze stap worden weggelaten, tenzij een ablation study aantoont dat deze specifiek voor *Warlords* wel winst oplevert.
* **Mock-API voor tests.** Hoewel nuttig voor CI en ontwikkelsnelheid, representeert deze niet volledig de echte omgeving. Een verbeterde versie zou aanvullende integratietests bevatten op een omgeving met beschikbare Atari-ROMs, om de realiteitsgetrouwheid van de tests te vergroten.


### Samenvatting van beslissingen

| # | Beslissing | Afgewezen alternatief | Belangrijkste reden |
|---|---|---|---|
| 1 | PPO-Clip | DQN, A2C, TRPO, SAC, MuZero | on-policy + stochastisch + goed in self-play (Schulman et al., 2017) |
| 2 | Gedeelde MLP over RAM | CNN over pixels; per-agent netten | RAM is compact (Anschel et al., 2017); agenten zijn symmetrisch (Gupta et al., 2017) |
| 3 | GAE | MC / TD / n-step | bias/variantie-knop via $\lambda$ (Schulman et al., 2016) |
| 4 | (T, N) buffer-layout | plat / env-majeur | matcht achterwaartse GAE-sweep (Huang et al., 2022) |
| 5 | HEPPO-standaardisaties | naïeve waardeverliesschaling | bekende PPO fragiliteit (Engstrom et al., 2020; Taha & Abdelhadi, 2025) |
| 6 | Voordeelstandaardisatie | geen | per-batch stabiliteit (Huang et al., 2022) |
| 7 | $\gamma=0.999, \lambda=0.97$ | reward shaping; curiosity | behoudt het werkelijke doel (Ng et al., 1999) |
| 8 | Self-play met pool | alle 4 slots trainen; co-evolutie | breekt moving-target (Berner et al., 2019) |
| 9 | ELO-gewogen sampling | uniform; alleen-laatste; PFSP | informatieve matchups; goedkoop |
| 10 | ELO als voortgangsmaat | gemiddelde return | normaliseert voor tegenstander (Elo, 1978; Silver et al., 2018) |

<a id="referenties"></a>
## Referenties

- Anschel, O., Baram, N., & Shimkin, N. (2017). Averaged-DQN. arXiv:1611.01929. https://arxiv.org/abs/1611.01929
- Bellemare, M. G., Naddaf, Y., Veness, J., & Bowling, M. (2013). The Arcade Learning Environment. arXiv:1207.4708. https://arxiv.org/abs/1207.4708
- Berner, C., Brockman, G., Chan, B., et al. (2019). Dota 2 with large scale deep reinforcement learning. arXiv:1912.06680. https://arxiv.org/abs/1912.06680
- CleanRL. (2023). *PPO — CleanRL implementation*. GitHub. https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/ppo.py
- DeepMind. (2019, 30 oktober). *AlphaStar: Mastering the real-time strategy game StarCraft II*. DeepMind Blog. https://deepmind.google/discover/blog/alphastar-mastering-the-real-time-strategy-game-starcraft-ii/
- Engstrom, L., Ilyas, A., Santurkar, S., Tsipras, D., Janoos, F., Rudolph, L., & Madry, A. (2020). Implementation matters in deep policy gradients. arXiv:2005.12729. https://arxiv.org/abs/2005.12729
- Farama Foundation. (2024, 14 juni). *Warlords — PettingZoo documentation*. https://pettingzoo.farama.org/environments/atari/warlords/
- Gupta, J. K., Egorov, M., & Kochenderfer, M. (2017). Cooperative multi-agent control using deep reinforcement learning. arXiv:1705.01965. https://arxiv.org/abs/1705.01965
- Heinrich, J., & Silver, D. (2016). Deep reinforcement learning from self-play in imperfect-information games. arXiv:1603.01121. https://arxiv.org/abs/1603.01121
- Huang, S., Dossa, F. J. D., Ye, C., et al. (2022). CleanRL. arXiv:2101.11061. https://arxiv.org/abs/2101.11061
- Huang, S., Ontañón, S., Cloppenburg, T., & Raffin, A. (2022). The 37 implementation details of PPO. ICLR Blog Track. https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/
- Hui, J. (2018, 12 december). *RL — Proximal Policy Optimization (PPO) explained*. Medium. https://jonathan-hui.medium.com/rl-proximal-policy-optimization-ppo-explained-77f014ec3f12
- Li, H., Yang, P., Liu, W., Yan, S., Zhang, X., & Zhu, D. (2025). Multi-Agent Reinforcement Learning in Games: Research and Applications. Biomimetics, 10(6), 375. https://doi.org/10.3390/biomimetics10060375
- Schulman, J., Levine, S., Moritz, P., Jordan, M., & Abbeel, P. (2015). Trust region policy optimization. arXiv:1502.05477. https://arxiv.org/abs/1502.05477
- Schulman, J., Moritz, P., Levine, S., Jordan, M., & Abbeel, P. (2016). High-dimensional continuous control using GAE. arXiv:1506.02438. https://arxiv.org/abs/1506.02438
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal policy optimization algorithms. arXiv:1707.06347. https://arxiv.org/abs/1707.06347
- Silver, D., Hubert, T., Schrittwieser, J., et al. (2018). A general reinforcement learning algorithm that masters chess, shogi, and Go. *Science, 362*(6419), 1140–1144. https://arxiv.org/abs/1712.01815
- Stable Baselines3. (2023). *RL tips and tricks*. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement learning: An introduction* (2nd ed.). MIT Press. http://incompleteideas.net/book/RLbook2020.pdf
- Taha, A., & Abdelhadi, A. (2025). HEPPO. arXiv:2501.12703. https://arxiv.org/abs/2501.12703
- Terry, J. K., Black, B., Grammel, N., et al. (2021). PettingZoo. arXiv:2009.14471. https://arxiv.org/abs/2009.14471
- Welford, B. P. (1962). Note on a method for calculating corrected sums of squares and products. *Technometrics, 4*(3), 419–420. https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance